In [ ]:
from delta.tables import DeltaTable
from pyspark.sql.functions import (
    col,
    concat_ws,
    dayofweek,
    hour,
    lit,
    round,
    sha2,
    unix_timestamp,
    when,
)
from pyspark.sql.types import DoubleType, IntegerType, TimestampType

# Read Bronze.
bronze_df = spark.table("nyc_taxi.bronze.green_taxi")

# Cast and standardize source types.
cast_rules = {
    "lpep_pickup_datetime": TimestampType(),
    "lpep_dropoff_datetime": TimestampType(),
    "VendorID": IntegerType(),
    "RatecodeID": IntegerType(),
    "payment_type": IntegerType(),
    "passenger_count": IntegerType(),
    "fare_amount": DoubleType(),
    "extra": DoubleType(),
    "mta_tax": DoubleType(),
    "improvement_surcharge": DoubleType(),
    "tip_amount": DoubleType(),
    "tolls_amount": DoubleType(),
    "total_amount": DoubleType(),
    "trip_distance": DoubleType(),
    "pickup_longitude": DoubleType(),
    "pickup_latitude": DoubleType(),
    "dropoff_longitude": DoubleType(),
    "dropoff_latitude": DoubleType(),
}

df = bronze_df
for column_name, data_type in cast_rules.items():
    if column_name in df.columns:
        df = df.withColumn(column_name, col(column_name).cast(data_type))
    else:
        df = df.withColumn(column_name, lit(None).cast(data_type))

# Normalize categorical values while preserving a valid downstream domain.
silver_df = (
    df.withColumn(
        "passenger_count",
        when(col("passenger_count").isNull() | (col("passenger_count") <= 0), lit(1))
        .when(col("passenger_count") > 6, lit(6))
        .otherwise(col("passenger_count")),
    )
    .withColumn(
        "payment_type",
        when(
            col("payment_type").isNull() | (col("payment_type") < 1) | (col("payment_type") > 6),
            lit(5),
        ).otherwise(col("payment_type")),
    )
    .withColumn(
        "RatecodeID",
        when(
            col("RatecodeID").isNull() | (col("RatecodeID") < 1) | (col("RatecodeID") > 6),
            lit(1),
        ).otherwise(col("RatecodeID")),
    )
)

# Add stable identifiers and derived analytics fields.
silver_df = (
    silver_df.withColumn(
        "trip_id",
        sha2(
            concat_ws(
                "||",
                col("lpep_pickup_datetime"),
                col("lpep_dropoff_datetime"),
                col("PULocationID"),
                col("DOLocationID"),
            ),
            256,
        ),
    )
    .withColumn(
        "trip_duration_minutes",
        round(
            (unix_timestamp(col("lpep_dropoff_datetime")) - unix_timestamp(col("lpep_pickup_datetime"))) / 60,
            2,
        ),
    )
    .withColumn(
        "fare_per_mile",
        when(col("trip_distance") > 0, round(col("fare_amount") / col("trip_distance"), 2)),
    )
    .withColumn(
        "tip_percentage",
        when(col("fare_amount") > 0, round((col("tip_amount") / col("fare_amount")) * 100, 2)).otherwise(lit(0.0)),
    )
    .withColumn("pickup_hour", hour(col("lpep_pickup_datetime")))
    .withColumn("pickup_day_of_week", dayofweek(col("lpep_pickup_datetime")))
    .withColumn(
        "anomaly_flag",
        when(col("trip_distance") > 100, lit("EXTREME_DISTANCE"))
        .when(col("fare_amount") > 500, lit("EXTREME_FARE"))
        .otherwise(lit("NORMAL")),
    )
)

# Exclude every repeated business key from Silver; all copies remain in quarantine.
duplicate_trip_ids = (
    silver_df.groupBy("trip_id")
    .count()
    .filter(col("count") > 1)
    .select("trip_id")
)

# Publish only valid analytical trips; zero-distance rows are retained in quarantine.
silver_df = (
    silver_df.filter(col("lpep_pickup_datetime").isNotNull())
    .filter(col("lpep_dropoff_datetime").isNotNull())
    .filter(col("lpep_dropoff_datetime") > col("lpep_pickup_datetime"))
    .filter(col("trip_distance") > 0)
    .filter(col("fare_amount") >= 0)
    .filter(col("total_amount") >= 0)
    .filter((col("trip_duration_minutes") > 0) & (col("trip_duration_minutes") <= 1440))
    .join(duplicate_trip_ids, "trip_id", "left_anti")
)

# Select the Silver contract.
silver_df = silver_df.select(
    "trip_id",
    col("VendorID").alias("vendor_id"),
    col("lpep_pickup_datetime").alias("pickup_datetime"),
    col("lpep_dropoff_datetime").alias("dropoff_datetime"),
    "trip_duration_minutes",
    "passenger_count",
    "trip_distance",
    "fare_per_mile",
    "PULocationID",
    "DOLocationID",
    col("RatecodeID").alias("rate_code_id"),
    "store_and_fwd_flag",
    "payment_type",
    "tip_amount",
    "tip_percentage",
    "fare_amount",
    "extra",
    "mta_tax",
    "improvement_surcharge",
    "tolls_amount",
    "total_amount",
    "trip_type",
    "pickup_hour",
    "pickup_day_of_week",
    "anomaly_flag",
)

table_name = "nyc_taxi.silver.green_taxi"
if spark.catalog.tableExists(table_name):
    if "anomaly_flag" not in spark.table(table_name).columns:
        spark.sql(f"ALTER TABLE {table_name} ADD COLUMNS (anomaly_flag STRING)")

    DeltaTable.forName(spark, table_name).alias("target").merge(
        silver_df.alias("source"),
        "target.trip_id = source.trip_id",
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
else:
    silver_df.write.format("delta").mode("overwrite").saveAsTable(table_name)
